## 1. Defining a Sample Corpus
Let's create a dummy corpus (Unchuncked Dataset) representing two longer documents. We will chunk these down into smaller searchable pieces.

In [10]:
sample_corpus = [
    {
        "metadata": {"source_id": "returns_policy.txt", "page": 0},
        "text": "At ShopEasy, we want you to be completely satisfied with your purchase. Customers can return most unused and unopened products within 30 days of delivery for a full refund. Refunds are processed to the original payment method within 5 to 7 business days after the return is received and approved by our warehouse team. Please note that customized items, perishable goods, and clearance merchandise are final sale and cannot be returned under any circumstances. If you receive a damaged item, you must report it to support within 48 hours of delivery."
    },
    {
        "metadata": {"source_id": "shipping_policy.txt", "page": 0},
        "text": "We offer several shipping options to meet your needs. Standard shipping usually takes 3-5 business days. Express delivery orders usually arrive within 24 to 48 hours depending on your zip code. Orders above 499 rupees qualify for free standard shipping automatically at checkout. For international orders, delivery times may vary between 7 to 14 business days depending on customs processing in the destination country. Duties and taxes for international shipments are the responsibility of the customer."
    }
]

## 2. Implement Chunking — Code and Metadata Together
Chunking and metadata assignment happen in **one step** — every chunk must carry a record of **where it came from** so your bot can eventually cite its sources.

In [11]:
def chunk_text(text, chunk_size=200, overlap=50): # Values lowered slightly from 500/75 for this demo to explicitly show the text splitting!
    """Strategy 1: fixed character windows with overlap."""
    if chunk_size <= overlap:
        raise ValueError("chunk_size must be larger than overlap")

    chunks = []
    start = 0
    while start < len(text):
        piece = text[start : start + chunk_size].strip()
        if piece:
            chunks.append(piece)
        start += chunk_size - overlap
    return chunks


def create_chunks_from_corpus(corpus, chunk_size=200, overlap=50):
    """Split every record; attach source_id, page, chunk_index; build stable ids."""
    all_chunks = []

    for record in corpus:
        text = record["text"]
        if not text:
            continue

        source_id = record["metadata"]["source_id"]
        page = record["metadata"]["page"]

        for chunk_index, chunk_body in enumerate(chunk_text(text, chunk_size, overlap)):
            all_chunks.append({
                "id": f"{source_id}__p{page}__c{chunk_index}",
                "text": chunk_body,
                "metadata": {
                    "source_id": source_id,
                    "page": page,
                    "chunk_index": chunk_index,
                },
            })

    return all_chunks

# Apply the chunking strategy to our sample corpus
chunks = create_chunks_from_corpus(sample_corpus, chunk_size=200, overlap=50)

print("Total chunks created:", len(chunks))
print("\n--- Sample Chunk Preview ---")
print("ID:", chunks[0]["id"])
print("Metadata:", chunks[0]["metadata"])
print("Text:", chunks[0]["text"])

Total chunks created: 8

--- Sample Chunk Preview ---
ID: returns_policy.txt__p0__c0
Metadata: {'source_id': 'returns_policy.txt', 'page': 0, 'chunk_index': 0}
Text: At ShopEasy, we want you to be completely satisfied with your purchase. Customers can return most unused and unopened products within 30 days of delivery for a full refund. Refunds are processed to th


## 3. Persist Chunked Documents in Chroma
Now we follow the same pipeline as the FAQ lab, but handling more rows and richer metadata.

In [16]:
import chromadb
from sentence_transformers import SentenceTransformer
from pprint import pprint

client=chromadb.PersistentClient("./Chroma_Store")

collection=client.get_or_create_collection(
    name="policy_chuncks",
    embedding_function=None
)

# Checing the collection 
print("NO of the collection made : ")
print(collection.count())
print("Collection : ")
pprint(collection.peek())

NO of the collection made : 
8
Collection : 
{'data': None,
 'documents': ['At ShopEasy, we want you to be completely satisfied with your '
               'purchase. Customers can return most unused and unopened '
               'products within 30 days of delivery for a full refund. Refunds '
               'are processed to th',
               'ery for a full refund. Refunds are processed to the original '
               'payment method within 5 to 7 business days after the return is '
               'received and approved by our warehouse team. Please note that '
               'customized ite',
               'ur warehouse team. Please note that customized items, '
               'perishable goods, and clearance merchandise are final sale and '
               'cannot be returned under any circumstances. If you receive a '
               'damaged item, you must',
               'umstances. If you receive a damaged item, you must report it '
               'to support within 48 hours

In [17]:
model=SentenceTransformer("all-MiniLM-L6-v2")

ids=[i["id"] for i in chunks]
documents=[i["text"] for i in chunks]
metadatas=[i["metadata"] for i in chunks]

embeddings=model.encode(
    documents,
    convert_to_numpy=True
).tolist()

collection.upsert(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=embeddings
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8904.37it/s]


In [18]:
# checking the collection
print("No of collection : ")
print(collection.count())
print("Collcetion Strucutre : ")
pprint(collection.peek())

No of collection : 
8
Collcetion Strucutre : 
{'data': None,
 'documents': ['At ShopEasy, we want you to be completely satisfied with your '
               'purchase. Customers can return most unused and unopened '
               'products within 30 days of delivery for a full refund. Refunds '
               'are processed to th',
               'ery for a full refund. Refunds are processed to the original '
               'payment method within 5 to 7 business days after the return is '
               'received and approved by our warehouse team. Please note that '
               'customized ite',
               'ur warehouse team. Please note that customized items, '
               'perishable goods, and clearance merchandise are final sale and '
               'cannot be returned under any circumstances. If you receive a '
               'damaged item, you must',
               'umstances. If you receive a damaged item, you must report it '
               'to support within 48 hour

## 4. Verify Storage and Run Semantic Search
Confirm the metadata survived the upsert by running `.get()`, and then test our chunking by asking a user query using `.query()`.

In [31]:
# Verify Storage 
print("----Spot Check----")
row=collection.get(ids=[ids[0]],include=["documents","metadatas"])
print("Metadata : ",row["metadatas"][0])
print("Documents : ",row["documents"][0][:105])



# 2. Semantic Search (Top-K query)
user_query = "How many days do I have to return a product?"
query_embeddings=model.encode(
    user_query,
    convert_to_numpy=True

).tolist()

results=collection.query(
    query_embeddings=query_embeddings,
    n_results=3
)

print("\n==========================")
print("Query:", user_query)
print("==========================")

for i in range(len(results["ids"][0])):
    print(f"\n🏆 Rank {i + 1}")
    print(" ID:", results["ids"][0][i])
    print(" Document:", results["documents"][0][i])
    print(" Metadata:", results["metadatas"][0][i])
    if results.get("distances"):
        print(" Distance:", results["distances"][0][i])

----Spot Check----
Metadata :  {'source_id': 'returns_policy.txt', 'chunk_index': 0, 'page': 0}
Documents :  At ShopEasy, we want you to be completely satisfied with your purchase. Customers can return most unused 

Query: How many days do I have to return a product?

🏆 Rank 1
 ID: returns_policy.txt__p0__c0
 Document: At ShopEasy, we want you to be completely satisfied with your purchase. Customers can return most unused and unopened products within 30 days of delivery for a full refund. Refunds are processed to th
 Metadata: {'chunk_index': 0, 'source_id': 'returns_policy.txt', 'page': 0}
 Distance: 0.6876528263092041

🏆 Rank 2
 ID: returns_policy.txt__p0__c1
 Document: ery for a full refund. Refunds are processed to the original payment method within 5 to 7 business days after the return is received and approved by our warehouse team. Please note that customized ite
 Metadata: {'chunk_index': 1, 'source_id': 'returns_policy.txt', 'page': 0}
 Distance: 0.9659268856048584

🏆 Rank 3
 I